# 11. Qwen3-VL で画像を判定する

このノートでは、画像をアップロードして日本語で質問し、Qwen3-VL に内容を判定してもらいます。物体・文字・状況の説明、比較、数え上げなどを試せます。

- モデル: `Qwen/Qwen3-VL-8B-Instruct`（L4で動作確認済み。30B版はVRAMに余裕のあるGPU向け）
- 量子化: bitsandbytes NF4 4bit。L4 の約22GB VRAMで動かすための設定です
- 画像入力: ノート下部のGradio UI、またはColab標準アップロードセルから画像と質問を入力
- ライセンス: モデルカード記載の Apache-2.0

**235B-A22B はさらに大きいモデルですが、4bitでも重みだけで約120GB規模となり、Colab Pro の通常の単一GPUでは動かせません。** このノートではL4で試せる大きなQwen3-VLを選んでいます。

## 実行手順

1. Colab の「ランタイム」→「ランタイムのタイプを変更」で GPU を選ぶ（L4推奨）
2. 上から順にセルを実行する
3. 最後の画面で画像をアップロードし、聞きたいことを書いて「Submit」
4. 初回は約17GBのモデルをダウンロードするため、時間と空きディスクが必要

> A100 でもこの4bitモデルを使えます。GPUメモリが不足した場合は、下の `MODEL_ID` を `Qwen/Qwen3-VL-8B-Instruct` に変えて、ランタイムを再起動して最初から実行してください。

## 1. GPU と実行環境を確認
GPU が割り当てられていない場合は、ランタイムのタイプを GPU に変更してから再実行してください。

In [ ]:
import torch, subprocess

if not torch.cuda.is_available():
    raise RuntimeError("GPU が見つかりません。Colab のランタイム設定で GPU を選び、ランタイムを再接続してください。")

gpu_name = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name} / VRAM: {vram_gib:.1f} GiB")
if vram_gib < 18:
    print("注意: このGPUでは8Bモデルを推奨します。30BモデルはVRAMが足りない可能性があります。")


## 2. 必要なライブラリをインストール
Qwen3-VL のサポートが入った Transformers と、4bit量子化・画像処理用ライブラリを入れます。インストール後にランタイム再起動が表示された場合は再起動し、セル1から続けてください。

In [ ]:
%pip -q install -U "transformers>=4.57.0" accelerate bitsandbytes qwen-vl-utils gradio


## 3. Qwen3-VL-8B-Instruct を4bitで読み込む
モデルの重みは初回のみ Hugging Face からダウンロードします。L4で安定して動かすため8Bを既定にしています。30B版はVRAMに余裕のあるGPU向けです。

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
# VRAMに余裕のあるGPUの場合: MODEL_ID = "Qwen/Qwen3-VL-30B-A3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()
print("読み込み完了:", MODEL_ID)
print("GPU使用量:", round(torch.cuda.memory_allocated() / 1024**3, 1), "GiB")

## 4. 画像判定関数を準備
質問欄を空欄にした場合は、画像の内容を日本語で説明するようにします。判定基準を具体的に書くと、より使いやすいよ。例:「この部品に割れや変形がありますか？見えた根拠も説明してください」

In [ ]:
from PIL import Image

def judge_image(image, question=""):
    if image is None:
        return "画像をアップロードしてください。"
    if not isinstance(image, Image.Image):
        image = Image.open(image)
    image = image.convert("RGB")
    question = (question or "").strip() or "この画像に写っているものと状況を日本語で説明してください。画像から確実に分かることと、推測を分けてください。"

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": question},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt"
    )
    inputs = inputs.to(model.device)
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=1024, do_sample=False)
    answer_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    return processor.decode(answer_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)

## 5. 画像アップロード画面を起動
Gradioの画像欄でアップロードが止まる場合は、下の「Colab標準アップロード」セルを使ってください。通常はGradioの入力欄に画像を置き、知りたいことを書いて「Submit」を押します。分類をしたい場合は、選択肢を指定した質問にすると判定結果を扱いやすくなります。

例:「画像の状態を『正常 / 傷あり / 判定不能』のどれかで答えてください。判断理由も一文で添えてください。」

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=judge_image,
    inputs=[
        gr.Image(type="pil", label="判定したい画像"),
        gr.Textbox(
            label="質問・判定基準（空欄なら画像を説明）",
            placeholder="例: 写っている製品に傷や汚れがあるか判定してください",
            lines=3,
        ),
    ],
    outputs=gr.Markdown(label="Qwen3-VL の回答"),
    title="Qwen3-VL 画像判定",
    description=f"モデル: {MODEL_ID} / 4bit量子化。画像の内容を確認して質問に回答します。",
    flagging_mode="never",
)
demo.launch(share=False, debug=False)

## 6. Colab標準アップロード（Gradioで画像を送れない場合）
画像欄でアップロードが止まる場合は、このセルを実行して「Choose Files」から画像を選んでください。


In [ ]:
from google.colab import files
from PIL import Image
from IPython.display import Markdown, display

uploaded_files = files.upload()
if uploaded_files:
    image_path = next(iter(uploaded_files))
    image = Image.open(image_path).convert("RGB")
    display(image)
    question = "この画像は何ですか？画像に見える特徴とあわせて日本語で説明してください。"
    answer = judge_image(image, question)
    display(Markdown(answer))


## 使うときのコツ

- 文字を読ませるときは、文字が鮮明に写る解像度の画像を使う
- 検品では、判定カテゴリ・判定基準・不明時の回答を質問文に明記する
- 画像の細部が小さいと見落とすことがあるため、結果は元画像でも確認する
- うまく読み込めない場合は、ランタイムを再起動し、8Bモデルに切り替える
- セッション終了後はGPUメモリが解放される。再接続後はセルを実行し直す

参考: [Qwen3-VL 公式リポジトリ](https://github.com/QwenLM/Qwen3-VL) / [30B-A3B モデルカード](https://huggingface.co/Qwen/Qwen3-VL-30B-A3B-Instruct) / [8B モデルカード](https://huggingface.co/Qwen/Qwen3-VL-8B-Instruct)